<a id='Libraries for Python 3.13.7'></a>
# Libraries for Python 3.13.7
List of libraries for Python 3.13.7 that we use:
- the [re](https://docs.python.org/3/library/re.html) library (for regular expression manipulation)
- the [ast](https://docs.python.org/3/library/ast.html) library (for abstract syntax tree manipulation)
- the [json](https://docs.python.org/3/library/json.html) library (for JSON file manipulation)
- the [openpyxl](https://pypi.org/project/openpyxl/) library (for reading Excel files)
- the [gradio](https://www.gradio.app/) library (for exposing ML models, APIs and more)
- the [date](https://docs.python.org/3/library/datetime.html#datetime.date) class (for date manipulation)
- the [timedelta](https://docs.python.org/3/library/datetime.html#datetime.timedelta) class (for date difference manipulation)
- the [cp_model](https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html) library (for implementing and solving CP-SAT models)
- the [get_column_letter](https://openpyxl.readthedocs.io/en/3.1/api/openpyxl.utils.cell.html#openpyxl.utils.cell.get_column_letter) function (for converting Excel column numbers)
- the [PatternFill](https://openpyxl.readthedocs.io/en/3.1/api/openpyxl.styles.fills.html#openpyxl.styles.fills.PatternFill) class (for decorating Excel cells)
- the [Font](https://openpyxl.readthedocs.io/en/3.1/api/openpyxl.styles.fonts.html#openpyxl.styles.fonts.Font) class (for editing Excel fonts)
- the [Alignment](https://openpyxl.readthedocs.io/en/3.1.3/api/openpyxl.styles.alignment.html) class (for manipulating Excel alignments)
- the [Border](https://openpyxl.readthedocs.io/en/3.1/api/openpyxl.styles.borders.html#openpyxl.styles.borders.Border) class (for manipulating Excel borders)
- the [Side](https://openpyxl.readthedocs.io/en/3.1/api/openpyxl.styles.borders.html#openpyxl.styles.borders.Side) class (for manipulating Excel borders)

In [3]:
import re
import ast
import json
import openpyxl
import gradio as gr
from datetime import date, timedelta
from ortools.sat.python import cp_model
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# Data extraction
In this section, we define the function that extracts data from a given Excel file and also auxiliary functions that transform some strings into suitable Python variables (lists and dictionnaries).

In [4]:
'''
Function that takes an Excel file as input, extracts public holidays, employee names and their absence dates,
and returns them formatted as JSON strings along with a shared state dictionary summarising the extracted data.

extract_data : Excel file ➜ (string, string, string, dictionary)
'''

def extract_data(fichier_excel):

    # Loading the active sheet from the Excel file
    
    ws = openpyxl.load_workbook(fichier_excel).active

    # Definition of the month dictionary (with or without accents, for safety)
    
    dict_mois = {'janvier': '01',
                 'février': '02', 'fevrier': '02',
                 'mars': '03',
                 'avril': '04',
                 'mai': '05',
                 'juin': '06',
                 'juillet': '07',
                 'août': '08', 'aout': '08',
                 'septembre': '09',
                 'octobre': '10',
                 'novembre': '11',
                 'décembre': '12', 'decembre': '12'}
    
    # Initialisation of the variables to fill or increment

    liste_feries = []
    dict_collab, dict_absences = {}, {}
    id, row = 1, 1

    while row <= ws.max_row:
        val = ws.cell(row=row, column=1).value

        # Search for rows containing a month name and a year number
        
        if isinstance(val, str):
            txt = val.lower().strip()
            mois = next((m for n, m in dict_mois.items() if n in txt), None)
            annee = next((a for a in txt.split() if a.isdigit()), None)

            # If a row contains a month and a year, we proceed to search for day numbers

            if mois and annee:
                row += 1
                jours = [(col, str(ws.cell(row=row, column=col).value).zfill(2))
                         for col in range(2, ws.max_column + 1)
                         if str(ws.cell(row=row, column=col).value).isdigit()]

                # Skipping the two rows of day numbers and day names

                row += 2

                # Adding public holidays to the "liste_feries" list

                for col, jour in jours:
                    val_cell = str(ws.cell(row=row, column=col).value).strip().upper()
                    if val_cell == 'JF':
                        liste_feries.append(f'{annee}-{mois}-{jour}')

                # Locating employee names and their absences

                while row <= ws.max_row:
                    nom = ws.cell(row=row, column=1).value
                    if not nom:
                        break

                    # Assigning a new ID in the case of a new employee
                    
                    if nom not in dict_collab:
                        dict_collab[nom] = id
                        id += 1

                    cid = dict_collab[nom]
                    dict_absences.setdefault(cid, [])

                    # Adding an employee's absence dates to the "dict_absences" dictionary

                    for col, jour in jours:
                        val_cell = str(ws.cell(row=row, column=col).value).strip().upper()
                        if val_cell == 'A':
                            dict_absences[cid].append(f'{annee}-{mois}-{jour}')

                    row += 1

        row += 1

    # Formatting public holidays, employee names and employee absences to JSON format

    json_feries = json.dumps(liste_feries, indent=0, ensure_ascii=False)
    json_collab = json.dumps({v: k for k, v in dict_collab.items()}, indent=0, ensure_ascii=False)
    json_absences = json.dumps(dict_absences, indent=0, ensure_ascii=False)

    # Definition of a state/dictionary of 5 variables shared between functions

    dict_state = {'annee': int(annee),
                  'json_feries': json_feries,
                  'json_collab': json_collab,
                  'json_absences': json_absences,
                  'nb_collaborateurs': len(dict_collab)}

    return json_feries, json_collab, json_absences, dict_state

In [5]:
'''
Function that extracts dates from a Python code containing a list of dates in the
"YYYY-MM-DD" format and returns a list of dates in Python format

string_to_list_dates : string ➜ list
'''

def string_to_list_dates(txt):

    return re.findall(r"'(\d{4}-\d{2}-\d{2})'", txt)

'''
Function that converts a string representing a dictionary into a
Python dictionary of names (string)

string_to_dict_names : string ➜ dictionary
'''

def string_to_dict_names(txt):
    
    data = ast.literal_eval(txt)
    
    return {int(k): str(v) for k, v in data.items()}

'''
Function that converts a string representing a dictionary into a
Python dictionary of integer lists

string_to_dict_list_int : string ➜ dictionary
'''

def string_to_dict_list_int(txt):
    
    data = ast.literal_eval(txt)

    return {int(k): list(map(int, v)) for k, v in data.items()}

'''
Function that converts a string representing a dictionary into a
Python dictionary of date lists

string_to_dict_list_dates : string ➜ dictionary
'''

def string_to_dict_list_dates(txt):
    
    data = ast.literal_eval(txt)
    
    return {int(k): list(v) for k, v in data.items()}

# CP-SAT optimisation model

## Model Creation
In this section, we create the CP-SAT model, as well as the necessary constraints for the scheduling problem. Detailed information on CP-SAT can be found in the following tutorials:
[https://developers.google.com/optimization/cp/cp_solver](https://developers.google.com/optimization/cp/cp_solver)<br>
[https://d-krupke.github.io/cpsat-primer/](https://d-krupke.github.io/cpsat-primer/)

In [ ]:
'''
Function that takes the list of working day indices, the employee unavailability dictionary,
the week index list and the date list as input, and builds a CP-SAT constraint programming
model with hard and soft constraints for permanence scheduling, returning the model along
with the permanence variables, non-consecutive duty day penalties and weekly excess penalties.

create_model : (list, dictionary, list, list) ➜ (CpModel, dictionary, list, list)
'''

def create_model(idx_jours_ouvrables, dict_indispo_collab, semaines, dates):

    nb_collaborateurs = len(dict_indispo_collab)
    nb_semaines = max(semaines) + 1

    # Creation of the blank CP-SAT model

    model = cp_model.CpModel()

    # Definition of a duty roster dictionary duty[e][d] (boolean variable)
    # duty[e][d] = 1 if employee e is on duty on working day d, and 0 otherwise

    permanence = {(e, d): model.NewBoolVar(f'permanence_e{e}_d{d}')
                 for e in range(1, nb_collaborateurs + 1) for d in idx_jours_ouvrables}

    # Definition of a weekly duty roster dictionary weekly_duty[e, w] (CP-SAT boolean variable)
    # weekly_duty[e, w] = 1 if employee e works a duty day during week w, and 0 otherwise

    permanence_semaine = {}
    for e in range(1, nb_collaborateurs + 1):
        for w in range(nb_semaines):
            permanence_semaine[e, w] = model.NewBoolVar(f'permanence_semaine_e{e}_w{w}')
            
            jours_semaine = [d for d in idx_jours_ouvrables if semaines[d] == w
                            and d not in dict_indispo_collab[e]]
            
            if jours_semaine:
                model.AddMaxEquality(permanence_semaine[e, w], [permanence[e, d] for d in jours_semaine])
            else:
                model.Add(permanence_semaine[e, w] == 0)

    # Hard constraint: every working day, exactly one employee is on duty

    [model.Add(sum(permanence[e, d] for e in range(1, nb_collaborateurs + 1)) == 1) for d in idx_jours_ouvrables]

    # Hard constraint: an employee can only be on duty on a working day if they are available

    [model.Add(permanence[e, d] == 0) for e in range(1, nb_collaborateurs + 1) for d in dict_indispo_collab[e]
    if d in idx_jours_ouvrables]

    # Hard constraint: for each employee, forbid more than 1 duty week every 4 weeks

    for e in range(1, nb_collaborateurs + 1):
        for w in range(nb_semaines - 3):
            model.Add(permanence_semaine[e, w] + permanence_semaine[e, w+1]
                     + permanence_semaine[e, w+2] + permanence_semaine[e, w+3] <= 1)

    # ========================================================================================
    # Soft constraint 1:
    # If an employee is on duty in a given week encourage them to do so on consecutive days
    # ========================================================================================

    # Definition of the list of non-consecutive duty days assigned to employees

    jours_non_consecutifs = []

    for e in range(1, nb_collaborateurs + 1):
        for w in range(nb_semaines):

            # Available days in the week for this employee
            
            jours_semaine = [d for d in idx_jours_ouvrables if semaines[d] == w
                            and d not in dict_indispo_collab[e]]
            
            # Only consider weeks with at least 2 working days

            for d1, d2 in zip(jours_semaine, jours_semaine[1:]):

                # truly consecutive days

                if dates[d2] == dates[d1] + timedelta(days=1):

                    # Definition of a boolean variable "non_consec" that is True if...
                    # ... an employee is on duty on exactly one of two consecutive days
                    
                    non_consec = model.NewBoolVar(f'non_consec_e{e}_w{w}_d{d1}')
                    model.Add(permanence[e, d1] + permanence[e, d2] == 1).OnlyEnforceIf(non_consec)
                    model.Add(permanence[e, d1] + permanence[e, d2] != 1).OnlyEnforceIf(non_consec.Not())
                    jours_non_consecutifs.append(non_consec)

    # ======================================================================================
    # Soft constraint 2: For each week, penalize if more than 1 employee is on duty
    # ======================================================================================

    # Definition of the list of the number of excess employees on duty each week

    liste_exces_semaine = []

    for w in range(nb_semaines):

        # Calculation of the list of the number of employees on duty each week

        collaborateurs_perm = []
        for e in range(1, nb_collaborateurs + 1):
            jours_semaine = [d for d in idx_jours_ouvrables if semaines[d] == w
                            and d not in dict_indispo_collab[e]]

            if jours_semaine:
                collab_perm = model.NewBoolVar(f'collab_perm_e{e}_w{w}')
                somme_semaine = sum(permanence[e, d] for d in jours_semaine)
                model.Add(somme_semaine >= 1).OnlyEnforceIf(collab_perm)
                model.Add(somme_semaine == 0).OnlyEnforceIf(collab_perm.Not())
                collaborateurs_perm.append(collab_perm)

        # Calculation of the excess number of employees on duty this week

        if collaborateurs_perm:
            nb_collab_perm = model.NewIntVar(0, len(collaborateurs_perm), f'nb_collab_perm_w{w}')
            model.Add(nb_collab_perm == sum(collaborateurs_perm))
            exces_semaine = model.NewIntVar(0, len(collaborateurs_perm), f'exces_semaine_w{w}')
            model.Add(exces_semaine >= nb_collab_perm - 1)
            liste_exces_semaine.append(exces_semaine)

    return model, permanence, jours_non_consecutifs, liste_exces_semaine

## Multi-criteria optimisation
In this section, we define the function that performs the optimisation, taking into the multiple constraints.

In [7]:
'''
Function that takes the CP-SAT model and its variables along with employee working day
dictionaries, unavailability data, solver parameters and penalty lists as input, defines
a weighted objective minimising fairness gaps, non-consecutive permanence days and weekly excess,
solves the model within a given time limit, and returns the solver, the solution status
and the number of permanence days assigned to each employee.

optimise_model : (dictionary, dictionary, list, dictionary, CpModel, list, list, list, int, int)
                 ➜ (CpSolver, status, dictionary)
'''

def optimise_model(dict_jours_collab,
                   permanence,
                   idx_jours_ouvrables,
                   dict_indispo_collab,
                   model,
                   semaines,
                   jours_non_consecutifs,
                   liste_exces_semaine,
                   temps_max,
                   nb_chercheurs):

    # Constants

    nb_collaborateurs = len(dict_jours_collab)
    nb_jours_ouvrables = len(idx_jours_ouvrables)
    nb_semaines = max(semaines) + 1

    # Definition of a target dictionary for the number of duty days per employee
    
    ratio_travail = {e: len(dict_jours_collab[e]) / 5 for e in range(1, nb_collaborateurs + 1)}

    cibles = {e: int(round(nb_jours_ouvrables * ratio_travail[e] / sum(ratio_travail.values())))
             for e in range(1, nb_collaborateurs + 1)}

    # Definition of a dictionary containing the number of duty days completed by each employee

    jours_permanence = {}

    for e in range(1, nb_collaborateurs + 1):
        jours_permanence[e] = model.NewIntVar(0, nb_jours_ouvrables, f'jours_permanence_e{e}')
        model.Add(jours_permanence[e] == sum(permanence[e, d] for d in idx_jours_ouvrables
                                            if d not in dict_indispo_collab[e]))

    # Definition of the variable "ecart_max" containing the maximum gap between...
    # ... each employee's duty target and their assigned duty days

    ecarts = []
    for e in range(1, nb_collaborateurs + 1):
        ecart_abs = model.NewIntVar(0, nb_jours_ouvrables, f'ecart_abs_e{e}')
        model.AddAbsEquality(ecart_abs, jours_permanence[e] - cibles[e])
        ecarts.append(ecart_abs)

    ecart_max = model.NewIntVar(0, nb_jours_ouvrables, 'ecart_max')
    model.AddMaxEquality(ecart_max, ecarts)

    # Definition of the variable "somme_ecarts" equal to the sum of gaps for each employee

    somme_ecarts = model.NewIntVar(0, nb_jours_ouvrables * nb_collaborateurs, 'somme_ecarts')
    model.Add(somme_ecarts == sum(ecarts))

    # Definition of the variable "nb_non_consecutifs" equal to the...
    # ... number of non-consecutive duty days assigned to employees

    nb_non_consecutifs = model.NewIntVar(0, nb_semaines * nb_collaborateurs * 5, 'nb_non_consecutifs')
    model.Add(nb_non_consecutifs == sum(jours_non_consecutifs))

    # Definition of the variable "nb_exces" equal to the sum (over all weeks)...
    # ... of the number of excess employees on duty each week

    nb_exces = model.NewIntVar(0, nb_semaines * nb_collaborateurs, 'nb_exces')
    model.Add(nb_exces == sum(liste_exces_semaine))

    # Definition of the objective equal to the minimisation of the sum of the 4 gaps from targets:
    # ecart_max + somme_ecarts + nb_non_consecutifs + 4*nb_exces
    # (by changing the factor in front of one of the 4 terms, we change the importance of that contribution)

    objectif = model.NewIntVar(0, nb_jours_ouvrables * 1000, 'objectif')
    model.Add(objectif == ecart_max + somme_ecarts + nb_non_consecutifs + 4*nb_exces)
    model.Minimize(objectif)

    # Parameters for solving the model
    # The larger num_search_workers is, the more configurations are explored
    # (but the search becomes non-deterministic if num_search_workers > 1)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = temps_max
    solver.parameters.num_search_workers = nb_chercheurs
    #solver.parameters.log_search_progress = True

    # Solving the model

    statut = solver.Solve(model)

    return solver, statut, jours_permanence

# Excel file creation
In this section, we define the function that generates an Excel file for the schedule selected by the optimisation model and also the function that, given the intut parameters, generates the outputs (the Excel file and a text message).

In [8]:
'''
Function that takes date lists, employee data, absence and public holiday information,
and a solved CP-SAT model as input, and generates a formatted Excel workbook containing
a colour-coded annual on-call schedule sheet organised by month as well as a summary
sheet with fairness statistics, attribution method explanations, public holidays and a
legend, returning the output filename and a status message if a solution was found, or
None and an error message otherwise.

create_excel : (list, dictionary, dictionary, list, dictionary, dictionary, dictionary,
               list, CpSolver, status, dictionary, dictionary) ➜ (string | None, string)
'''

def create_excel(dates,
                 date_vers_index,
                 indices_dict_absences,
                 liste_feries,
                 dict_collab,
                 dict_jours_collab,
                 dict_absences,
                 idx_jours_ouvrables,
                 solver,
                 statut,
                 permanence,
                 jours_permanence):

    if statut == cp_model.OPTIMAL or statut == cp_model.FEASIBLE:
        
        # Creating easily definable variables inside the function

        annee = dates[0].year
        date_debut = date(annee, 1, 1)
        nombre_jours = (date(annee, 12, 31) - date(annee, 1, 1)).days + 1

        # Definition of abbreviated weekdays, weekdays,...

        jours_semaine_court = ['lun', 'mar', 'mer', 'jeu', 'ven', 'sam', 'dim']
        jours_semaine_long = ['lundi', 'mardi', 'mercredi', 'jeudi', 'vendredi', 'samedi', 'dimanche']

        est_weekend = [d.weekday() >= 5 for d in dates]
        idx_liste_feries = {(date.fromisoformat(jour) - date_debut).days for jour in liste_feries}
        mois = ['janvier', 'février', 'mars', 'avril', 'mai', 'juin', 'juillet', 'août', 'septembre', 'octobre', 'novembre', 'décembre']

        nb_collaborateurs = len(dict_collab)
        nb_jours_ouvrables = len(idx_jours_ouvrables)
        ratio_travail = {e: len(dict_jours_collab[e]) / 5 for e in range(1, nb_collaborateurs + 1)}

        cibles = {e: int(round(nb_jours_ouvrables * ratio_travail[e] / sum(ratio_travail.values())))
                 for e in range(1, nb_collaborateurs + 1)}

        fichier_sortie = f'planning_collaborateurs_{annee}.xlsx'
        
        # Style configuration
        
        STYLES = {
            'entete_mois': {'fill': PatternFill(start_color='203864', end_color='203864', fill_type='solid'),
                            'font': Font(bold=True, color='FFFFFF', size=12)},
            'entete_date': {'fill': PatternFill(start_color='366092', end_color='366092', fill_type='solid'),
                            'font': Font(bold=True, color='FFFFFF', size=10)},
            'entete_jour': {'fill': PatternFill(start_color='5B9BD5', end_color='5B9BD5', fill_type='solid'),
                            'font': Font(bold=True, color='FFFFFF', size=9)},
            'entete_collab': {'fill': PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid'),
                              'font': Font(bold=True, color='FFFFFF', size=10)},
            'permanence': {'fill': PatternFill(start_color='FFC000', end_color='FFC000', fill_type='solid'),
                           'font': Font(bold=True)},
            'disponible': {'fill': PatternFill(start_color='C6E0B4', end_color='C6E0B4', fill_type='solid'),
                           'font': Font()},
            'absence': {'fill': PatternFill(start_color='F4B084', end_color='F4B084', fill_type='solid'),
                        'font': Font()},
            'weekend': {'fill': PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid'),
                        'font': Font(color='808080', bold=True, size=9)},
            'jour_ferie': {'fill': PatternFill(start_color='E7E6E6', end_color='E7E6E6', fill_type='solid'),
                           'font': Font(color='D32F2F', bold=True, size=9)}
        }
        
        bordure_fine = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'),
                              bottom=Side(style='thin'))
        
        # Utility functions

        def appliquer_style(cellule, style_key, valeur=None):
            if valeur is not None:
                cellule.value = valeur
            cellule.fill = STYLES[style_key]['fill']
            cellule.font = STYLES[style_key]['font']
            cellule.alignment = Alignment(horizontal='center', vertical='center')
            cellule.border = bordure_fine
        
        def determiner_etat_cellule(collab_id, jour_id):
            """Retourne (valeur, style) pour une cellule donnée"""
            if jour_id in idx_liste_feries:
                return 'JF', 'jour_ferie'
            if est_weekend[jour_id]:
                return '', 'weekend'
            if jour_id in indices_dict_absences[collab_id]:
                return 'A', 'absence'
            if dates[jour_id].weekday() not in dict_jours_collab[collab_id]:
                return '', 'weekend'
            
            # Permanence check

            if collab_id <= nb_collaborateurs and jour_id in idx_jours_ouvrables:
                if solver.Value(permanence[collab_id, jour_id]) == 1:
                    return 'P', 'permanence'
            
            return 'D', 'disponible'
        
        def creer_entete_mois(feuille, ligne, nom_mois, nb_jours):
            """Crée l'en-tête du mois fusionné"""
            cellule = feuille.cell(row=ligne, column=1)
            appliquer_style(cellule, 'entete_mois', nom_mois)
            feuille.merge_cells(start_row=ligne, start_column=1, 
                            end_row=ligne, end_column=nb_jours + 1)
        
        def creer_ligne_dates(feuille, ligne, dates_mois):
            """Crée la ligne des dates du mois"""
            appliquer_style(feuille.cell(row=ligne, column=1), 'entete_collab', 'collaborateur')
            for idx_col, d in enumerate(dates_mois, start=2):
                appliquer_style(feuille.cell(row=ligne, column=idx_col), 'entete_date', d.strftime('%d'))
        
        def creer_ligne_jours(feuille, ligne, dates_mois, indices_mois):
            """Crée la ligne des jours de la semaine"""
            appliquer_style(feuille.cell(row=ligne, column=1), 'entete_collab', '')
            
            for idx_col, (d, jour_id) in enumerate(zip(dates_mois, indices_mois), start=2):
                cellule = feuille.cell(row=ligne, column=idx_col)
                jour_abr = jours_semaine_court[d.weekday()]
                
                if jour_id in idx_liste_feries:
                    appliquer_style(cellule, 'jour_ferie', jour_abr)
                elif d.weekday() >= 5:
                    appliquer_style(cellule, 'weekend', jour_abr)
                else:
                    appliquer_style(cellule, 'entete_jour', jour_abr)
        
        def creer_ligne_collaborateur(feuille, ligne, collab_id, dates_mois, indices_mois):
            """Crée une ligne pour un collaborateur"""
            cellule_nom = feuille.cell(row=ligne, column=1)
            cellule_nom.value = dict_collab[collab_id]
            cellule_nom.fill = STYLES['entete_collab']['fill']
            cellule_nom.font = STYLES['entete_collab']['font']
            cellule_nom.alignment = Alignment(horizontal='left', vertical='center')
            cellule_nom.border = bordure_fine
            
            for idx_col, jour_id in enumerate(indices_mois, start=2):
                cellule = feuille.cell(row=ligne, column=idx_col)
                valeur, style = determiner_etat_cellule(collab_id, jour_id)
                appliquer_style(cellule, style, valeur)
        
        # Data preparation

        donnees_mois = {}
        for d in range(nombre_jours):
            cle_mois = dates[d].month
            if cle_mois not in donnees_mois:
                donnees_mois[cle_mois] = {
                    'nom': f"{mois[dates[d].month - 1]} {dates[d].year}",
                    'dates': [],
                    'indices': []
                }
            donnees_mois[cle_mois]['dates'].append(dates[d])
            donnees_mois[cle_mois]['indices'].append(d)
        
        # Workbook creation

        classeur = openpyxl.Workbook()
        feuille = classeur.active
        feuille.title = f'Planning Annuel {annee}'
        
        ligne_actuelle = 1
        
        # Schedule generation by month

        for num_mois in range(1, 13):
            if num_mois not in donnees_mois:
                continue
            
            info_mois = donnees_mois[num_mois]
            dates_mois = info_mois['dates']
            indices_mois = info_mois['indices']
            
            creer_entete_mois(feuille, ligne_actuelle, info_mois['nom'], len(dates_mois))
            ligne_actuelle += 1
            
            creer_ligne_dates(feuille, ligne_actuelle, dates_mois)
            ligne_actuelle += 1
            
            creer_ligne_jours(feuille, ligne_actuelle, dates_mois, indices_mois)
            ligne_actuelle += 1
            
            for collab_id in range(1, nb_collaborateurs + 1):
                creer_ligne_collaborateur(feuille, ligne_actuelle, collab_id, dates_mois, indices_mois)
                ligne_actuelle += 1
            
            ligne_actuelle += 1
        
        # Column configuration

        feuille.column_dimensions['A'].width = 20
        for idx_col in range(2, 33):
            feuille.column_dimensions[get_column_letter(idx_col)].width = 4.5
        feuille.freeze_panes = 'B1'
        
        # Summary sheet

        feuille_resume = classeur.create_sheet(title='Résumé')
        
        donnees_resume = [['Collaborateur', 'Ratio Temps Plein', 
                        'Jours Permanence', 'Cible', 'Écart', 'Statut']]
        
        for e in range(1, nb_collaborateurs + 1):
            ratio = ratio_travail[e]
            nb_jours_perm = solver.Value(jours_permanence[e])
            cible = cibles[e]
            ecart = nb_jours_perm - cible
            statut_emploi = "⚠ Temps partiel" if ratio < 0.9 else "Temps plein"
            
            donnees_resume.append([
                dict_collab[e], f"{ratio:.1%}",
                nb_jours_perm, cible, f"{ecart:+d}", statut_emploi
            ])
        
        # Filling in the summary sheet

        for idx_ligne, donnees_ligne in enumerate(donnees_resume, start=1):
            for idx_col, valeur in enumerate(donnees_ligne, start=1):
                cellule = feuille_resume.cell(row=idx_ligne, column=idx_col)
                cellule.value = valeur
                cellule.border = bordure_fine
                cellule.alignment = Alignment(horizontal='center', vertical='center')
                
                if idx_ligne == 1:
                    appliquer_style(cellule, 'entete_date')
                elif idx_ligne > 1 and idx_col == 6 and 'Temps partiel' in str(valeur):
                    cellule.fill = PatternFill(start_color='FFE699', end_color='FFE699', fill_type='solid')
        
        for col in range(1, 7):
            feuille_resume.column_dimensions[get_column_letter(col)].width = 22
        
        # Attribution method explanations

        ligne = len(donnees_resume) + 3
        feuille_resume.cell(row=ligne, column=1).value = 'Méthode d\'attribution'
        feuille_resume.cell(row=ligne, column=1).font = Font(bold=True, size=12)
        ligne += 1
        
        explications = [
            "1. Les permanences sont attribuées proportionnellement au temps de travail",
            "2. Seuil temps partiel: 90% d'un temps plein",
            "3. Contrainte souple 1: Favoriser la continuité des permanences (jours consécutifs)",
            "4. Contrainte souple 2: Favoriser au plus 1 collaborateur de permanence par semaine",
            "5. Contrainte stricte: Maximum 1 semaine de permanence toutes les 4 semaines par collaborateur"
        ]
        
        for expl in explications:
            feuille_resume.cell(row=ligne, column=1).value = expl
            feuille_resume.cell(row=ligne, column=1).alignment = Alignment(horizontal='left', vertical='center')
            feuille_resume.merge_cells(start_row=ligne, start_column=1, end_row=ligne, end_column=4)
            ligne += 1
        
        # List of public holidays

        ligne += 2
        feuille_resume.cell(row=ligne, column=1).value = f"Jours fériés {annee} (Etat de Genève)"
        feuille_resume.cell(row=ligne, column=1).font = Font(bold=True, size=12)
        ligne += 1
        
        for jour_ferie_str in liste_feries:
            if jour_ferie_str in date_vers_index:
                idx = date_vers_index[jour_ferie_str]
                date_obj = dates[idx]
                jour_nom = jours_semaine_long[date_obj.weekday()]
                feuille_resume.cell(row=ligne, column=1).value = f"{date_obj.strftime('%Y-%m-%d')} ({jour_nom})"
                ligne += 1
        
        # Legend

        ligne += 2
        feuille_resume.cell(row=ligne, column=1).value = 'Légende'
        feuille_resume.cell(row=ligne, column=1).font = Font(bold=True, size=12)
        ligne += 1
        
        donnees_legende = [
            ('P', 'Permanence', 'permanence'),
            ('D', 'Disponible', 'disponible'),
            ('A', 'Absence', 'absence'),
            ('JF', 'Jour férié (Etat de Genève)', 'jour_ferie'),
            ('', 'Week-end ou jour non travaillé', 'weekend')
        ]
        
        for code, desc, style in donnees_legende:
            feuille_resume.cell(row=ligne, column=1).value = code
            feuille_resume.cell(row=ligne, column=2).value = desc
            appliquer_style(feuille_resume.cell(row=ligne, column=1), style)
            feuille_resume.cell(row=ligne, column=2).alignment = Alignment(horizontal='left', vertical='center')
            ligne += 1

        classeur.save(fichier_sortie)

        message = f"✅ Solution trouvée (statut: {'OPTIMAL' if statut == cp_model.OPTIMAL else 'FEASIBLE'})"

        return fichier_sortie, message

    else:
        message = f'❌ Aucune solution trouvée  (statut: {solver.StatusName(statut)})'
        
        return None, message

In [9]:
'''
Function that takes a year, four JSON strings for public holidays, employee names,
working days and absences, along with solver parameters as input, parses all inputs,
builds the calendar and availability data, calls create_model, optimise_model and
create_excel in sequence, and returns the output Excel filename and a status message.

create_result : (int, string, string, string, string, int, int) ➜ (string | None, string)
'''

def create_result(annee, txt1, txt2, txt3, txt4, temps_max, nb_chercheurs):

    liste_feries = string_to_list_dates(txt1)
    dict_collab = string_to_dict_names(txt2)
    dict_jours_collab = string_to_dict_list_int(txt3)
    dict_absences = string_to_dict_list_dates(txt4)

    # Constants

    nombre_jours = (date(annee, 12, 31) - date(annee, 1, 1)).days + 1
    nb_collaborateurs = len(dict_collab)

    # Definition of the first day of the year, the set of all dates in the year and...
    # ... the dictionary containing the date and index of each day of the year

    date_debut = date(annee, 1, 1)
    dates = [date_debut + timedelta(days=i) for i in range(nombre_jours)]
    date_vers_index = {d.strftime('%Y-%m-%d'): i for i, d in enumerate(dates)}

    # Definition of weekend dates

    est_weekend = [d.weekday() >= 5 for d in dates]

    # Definition of the set of public holiday indices,...
    # of the list of working day indices and number of working days

    idx_liste_feries = {(date.fromisoformat(jour) - date_debut).days for jour in liste_feries}
    idx_jours_ouvrables = [d for d in range(nombre_jours) if not est_weekend[d] and d not in idx_liste_feries]

    # Definition of the "semaines" list containing the week index for each day of the year
    # Equals 0 before the first Sunday of the year, then (1 + number of elapsed days modulo 7) afterwards

    premier_dimanche = next(d for d in dates if d.weekday() == 6)

    semaines = [0 if d <= premier_dimanche
                else 1 + ((d - (premier_dimanche + timedelta(days=1))).days // 7) for d in dates]

    # Definition of an absence dictionary for each employee

    indices_dict_absences = {i: {date_vers_index[d] for d in dates_absence if d in date_vers_index}
                            for i, dates_absence in dict_absences.items()}

    # Definition of a dictionary of unavailable days of the year for each employee

    dict_indispo_collab = {
        e: {d for d, date_obj in enumerate(dates) if date_obj.weekday() not in dict_jours_collab[e]}
        | indices_dict_absences[e] for e in range(1, nb_collaborateurs + 1)}

    model, permanence, jours_non_consecutifs, liste_exces_semaine = create_model(idx_jours_ouvrables,
                                                                                 dict_indispo_collab,
                                                                                 semaines,
                                                                                 dates)

    solver, statut, jours_permanence = optimise_model(dict_jours_collab,
                                                      permanence,
                                                      idx_jours_ouvrables,
                                                      dict_indispo_collab,
                                                      model,
                                                      semaines,
                                                      jours_non_consecutifs,
                                                      liste_exces_semaine,
                                                      temps_max,
                                                      nb_chercheurs)

    fichier, message = create_excel(dates,
                                    date_vers_index,
                                    indices_dict_absences,
                                    liste_feries,
                                    dict_collab,
                                    dict_jours_collab,
                                    dict_absences,
                                    idx_jours_ouvrables,
                                    solver,
                                    statut,
                                    permanence,
                                    jours_permanence)
    
    return fichier, message

# Gradio interface
In this section, we construct the Gradio interface for the solution and the necessary functions to run it.

In [10]:
'''
Function that takes the number of employees as input and returns the string representing
the python dictionary for the working days (from 0 to 4) of each employee. This
string can then be modified by the user.

build_jours_travail : integer ➜ string
'''

def build_jours_travail(nb_collaborateurs):
    result_string = '{\n'
    for i in range(1, nb_collaborateurs + 1):
        result_string += f'"{i}": [0, 1, 2, 3, 4],\n'
    
    return result_string + '}'

'''
Function that takes a dictionary state of length 5 as input and returns the 5 values
of the dictionary if it is non-empty (except for value 4 where it returns build_jours_travail(value)),
and 5 times "None" otherwise.

on_tab_select : dictionary ➜ 5-tuple
'''

def on_tab_select(dict_state):
    if dict_state is None:
        return None, None, None, None, None
        
    return (dict_state['annee'],
            dict_state['json_feries'],
            dict_state['json_collab'],
            build_jours_travail(dict_state['nb_collaborateurs']),
            dict_state['json_absences'])

In [11]:
# ==================
# Gradio interface
# ==================

with gr.Blocks() as demo:

    # Definition of a shared state between tabs

    dict_state = gr.State()

    # --------------------------------------------------------------
    # Tab 1: Data extraction from an Excel file
    # --------------------------------------------------------------

    with gr.Tab(label='Extraction des données du fichier Excel'):
        with gr.Row():
            
            # Creating buttons in the left column
            
            with gr.Column():
                file_input = gr.File(label='Fichier Excel', file_types=['.xlsx'])
                extract_btn = gr.Button('Extraire données')
                reset_btn_tab1 = gr.Button('Réinitialiser')

            # Output of extracted data in the right column

            with gr.Column():
                feries_out = gr.Code(label='Jours fériés Genève (liste de dates)')
                collab_out = gr.Code(label='Nom des collaborateurs (ID ➜ Nom)')
                absences_out = gr.Code(label='Absences des collaborateurs (ID ➜ liste de dates)')

        # Definition of button actions when activated

        extract_btn.click(fn=extract_data,
                          inputs=file_input,
                          outputs=[feries_out, collab_out, absences_out, dict_state])

        reset_btn_tab1.click(fn=lambda: (None, None, None, None, None),
                             inputs=[],
                             outputs=[file_input, feries_out, collab_out, absences_out, dict_state])

    # -------------------------------------------------
    # Tab 2: Creation of the Excel planning file
    # -------------------------------------------------

    with gr.Tab(label='Création du planning des permanences') as tab2:

        with gr.Row():

            # Input values in the left column
            
            with gr.Column():
                annee_input = gr.Number(label='Année', precision=0)
                jours_feries_input = gr.Textbox(label='Jours fériés Genève (liste de dates)', lines=2)
                noms_collab_input = gr.Textbox(label='Nom des collaborateurs (ID ➜ Nom)', lines=2)
                jours_travail_input = gr.Textbox(label='Jours travaillés par chaque collaborateur (ID ➜ liste de jours de la semaine)', lines=2)
                jours_absence_input = gr.Textbox(label='Absences des collaborateurs (ID ➜ liste de dates)', lines=2)


            with gr.Column():

                # Input values in the right column

                temps_max_input = gr.Number(label='Temps de recherche maximal (en secondes)', precision=0, value=30)
                nb_chercheurs_input = gr.Number(label='Nombre de chercheurs en parallèle', precision=0, value=16)

                # Output of results in the right column

                result_text = gr.Textbox(label='Résultat')
                result_file = gr.File(label='xlsx', file_types=['.xlsx'])

                # Creating buttons in the right column

                create_btn = gr.Button('Créer planning')
                reset_btn_tab2 = gr.Button('Réinitialiser')

        # Definition of button actions when activated

        create_btn.click(fn=create_result,
                         inputs=[annee_input,
                                 jours_feries_input,
                                 noms_collab_input,
                                 jours_travail_input,
                                 jours_absence_input,
                                 temps_max_input,
                                 nb_chercheurs_input],
                         outputs=[result_file, result_text])

        reset_btn_tab2.click(fn=lambda temps, nombre: (None, None, None, None, None, temps, nombre, None, None),
                             inputs=[temps_max_input, nb_chercheurs_input],
                             outputs=[annee_input,
                                      jours_feries_input,
                                      noms_collab_input,
                                      jours_travail_input,
                                      jours_absence_input,
                                      temps_max_input,
                                      nb_chercheurs_input,
                                      result_text,
                                      result_file])

        # Execution of auto-fill when selecting tab 2

        tab2.select(fn=on_tab_select,
                    inputs=dict_state,
                    outputs=[annee_input,
                             jours_feries_input,
                             noms_collab_input,
                             jours_travail_input,
                             jours_absence_input])

demo.launch(theme=gr.themes.Base())


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
